# Video-driven VRM expressions
Predict both eye states and the mouth state, then play them in sync with the source video. The head and body stay fixed.
Run the cells in order. Set the paths in the first setup cell. In the tracking window, press 1, 2 or 3 to adjust the screen-left eye, screen-right eye or mouth. Include both lips and some space for the mouth to open.
The eyes use the original template method. The mouth uses a fixed region. An unknown state returns the expression to neutral.

In [9]:
from pathlib import Path
from datetime import datetime
from functools import partial
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from threading import Thread
import csv
import json
import pickle
import webbrowser

import cv2
import numpy as np

In [10]:
# Set these three paths. Use separate trained models for the eyes and mouth.
PROJECT_DIR = Path(r"C:\Users\user\Desktop\M2\Computer Vision and Image Processing\Major project")
VIDEO_PATH = PROJECT_DIR / "Michelle C. Smith.mp4"
EYE_MODEL_PATH = PROJECT_DIR / "eye_training_runs" / "20261026_213429_a820cf" / "eye_hog_svm.pkl"
MOUTH_MODEL_PATH = PROJECT_DIR / "mouth_viseme_training_runs" / "20261026_172858_d864ec" / "mouth_viseme_hog_svm.pkl"

# Set True to save a debug video with boxes. The notebook always saves CSV, JSON and run settings.
SAVE_DEBUG_VIDEO = True
FEATURE_PARAMS = dict(maxCorners=150, qualityLevel=0.01, minDistance=7, blockSize=7)
LK_PARAMS = dict(winSize=(21, 21), maxLevel=3,
                 criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
MIN_POINTS = 10
FB_THRESHOLD = 1.5
RANSAC_THRESHOLD = 3.0
REPLENISH_THRESHOLD = 80
SEARCH_MARGIN = 0.35
MATCH_THRESHOLD = 0.15

for path in (VIDEO_PATH, EYE_MODEL_PATH, MOUTH_MODEL_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"Set the correct path first: {path}")
print("Paths checked. OpenCV:", cv2.__version__)


Paths checked. OpenCV: 4.10.0


In [11]:
def select_roi(frame, title):
    """Select a region manually."""

    x, y, w, h = cv2.selectROI(
        title, frame,
        showCrosshair=True,
        fromCenter=False
    )
    cv2.destroyWindow(title)

    if w < 8 or h < 8:
        raise ValueError("Region is too small or selection was canceled.")

    return int(x), int(y), int(w), int(h)

def rect_points(rect):
    x, y, w, h = rect

    return np.float32([
        [x, y],
        [x + w - 1, y],
        [x + w - 1, y + h - 1],
        [x, y + h - 1]
    ]).reshape(-1, 1, 2)

def track_points(previous_gray, current_gray, points, reference):
    """Check forward-backward flow. reference keeps the initial coordinates."""

    current, status, _ = cv2.calcOpticalFlowPyrLK(
        previous_gray, current_gray,
        points, None, **LK_PARAMS
    )

    if current is None or status is None:
        return None, None

    valid = (
        (status.ravel() == 1)
        & np.isfinite(current.reshape(-1, 2)).all(axis=1)
    )

    if not valid.any():
        return None, None

    old = points[valid]
    current = current[valid]
    reference = reference[valid]

    backward, status_back, _ = cv2.calcOpticalFlowPyrLK(
        current_gray, previous_gray,
        current, None, **LK_PARAMS
    )

    if backward is None or status_back is None:
        return None, None

    error = np.linalg.norm(
        old.reshape(-1, 2) - backward.reshape(-1, 2),
        axis=1
    )

    height, width = current_gray.shape
    xy = current.reshape(-1, 2)

    reliable = (
        (status_back.ravel() == 1)
        & np.isfinite(error)
        & (error < FB_THRESHOLD)
        & (xy[:, 0] >= 0)
        & (xy[:, 0] < width)
        & (xy[:, 1] >= 0)
        & (xy[:, 1] < height)
    )

    return current[reliable], reference[reliable]

def replenish(gray, points, reference, transform, corners):
    """Add corners for global tracking. These points have no facial labels."""

    if len(points) >= REPLENISH_THRESHOLD:
        return points, reference

    mask = np.zeros(gray.shape, dtype=np.uint8)

    cv2.fillConvexPoly(
        mask,
        np.rint(corners).astype(np.int32).reshape(-1, 2),
        255
    )

    mask = cv2.erode(
        mask, np.ones((9, 9), dtype=np.uint8)
    )

    for point in points.reshape(-1, 2):
        cv2.circle(
            mask,
            tuple(np.rint(point).astype(int)),
            10, 0, -1
        )

    params = FEATURE_PARAMS.copy()
    params["maxCorners"] = (
        FEATURE_PARAMS["maxCorners"] - len(points)
    )

    if params["maxCorners"] <= 0:
        return points, reference

    new_points = cv2.goodFeaturesToTrack(
        gray, mask=mask, **params
    )

    if new_points is None:
        return points, reference

    new_reference = cv2.perspectiveTransform(
        new_points, np.linalg.inv(transform)
    )

    valid = np.isfinite(
        new_reference.reshape(-1, 2)
    ).all(axis=1)

    return (
        np.concatenate([points, new_points[valid]], axis=0),
        np.concatenate([reference, new_reference[valid]], axis=0)
    )

def locate_eye(aligned, valid_mask, mouth):
    """Match the local template. Use the global motion estimate if matching fails."""

    x, y, w, h = mouth["rect"]
    height, width = aligned.shape

    mx = max(4, int(w * SEARCH_MARGIN))
    my = max(4, int(h * SEARCH_MARGIN))

    x0, y0 = max(0, x - mx), max(0, y - my)
    x1 = min(width, x + w + mx)
    y1 = min(height, y + h + my)

    search = aligned[y0:y1, x0:x1]
    rect = mouth["rect"]
    score = float("inf")
    matched = False

    if search.shape[0] >= h and search.shape[1] >= w:
        response = cv2.matchTemplate(
            search,
            mouth["template"],
            cv2.TM_SQDIFF_NORMED
        )

        valid_search = (
            valid_mask[y0:y1, x0:x1] > 250
        ).astype(np.float32)

        coverage = cv2.matchTemplate(
            valid_search,
            np.ones((h, w), dtype=np.float32),
            cv2.TM_CCORR
        ) / float(w * h)

        response[coverage < 0.999] = np.inf
        response[~np.isfinite(response)] = np.inf

        if np.isfinite(response).any():
            score, _, location, _ = cv2.minMaxLoc(response)

            if score <= MATCH_THRESHOLD:
                rect = (
                    x0 + location[0],
                    y0 + location[1],
                    w, h
                )
                matched = True

    return rect, float(score), matched


def load_expression_model(path, size_key):
    with open(path, "rb") as file:
        bundle = pickle.load(file)
    model = bundle["model"]
    size = tuple(bundle[size_key])
    hog = cv2.HOGDescriptor(**bundle["hog_params"])
    labels = {int(value): key for key, value in bundle["label_to_id"].items()}
    if set(labels.values()) != {"open", "closed"}:
        raise ValueError(f"Unknown class mapping: {labels}")
    if hog.getDescriptorSize() != model.n_features_in_:
        raise ValueError(f"HOG feature size does not match the model: {path}")
    return model, size, hog, labels


def classify_region(crop, model_info):
    model, size, hog, labels = model_info
    interpolation = (cv2.INTER_AREA
                     if crop.shape[1] >= size[0] and crop.shape[0] >= size[1]
                     else cv2.INTER_LINEAR)
    image = cv2.resize(crop, size, interpolation=interpolation)
    feature = hog.compute(image).reshape(1, -1).astype(np.float32)
    return labels[int(model.predict(feature)[0])]


def initialize_expression_tracking(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    head = select_roi(frame, "Select whole head, then Enter")
    regions = []
    for name in ("screen_left_eye", "screen_right_eye", "mouth"):
        rect = select_roi(frame, f"Select {name}, then Enter")
        x, y, w, h = rect
        regions.append({"name": name, "rect": rect,
                        "template": gray[y:y+h, x:x+w].copy()})
    mask = np.zeros(gray.shape, np.uint8)
    x, y, w, h = head
    mask[y:y+h, x:x+w] = 255
    points = cv2.goodFeaturesToTrack(gray, mask=mask, **FEATURE_PARAMS)
    if points is None or len(points) < MIN_POINTS:
        raise ValueError("Too few head tracking points. Select the region again.")
    return gray, points, rect_points(head), regions


In [12]:
# Process both eyes and the mouth together to keep frame indices in sync.
eye_info = load_expression_model(EYE_MODEL_PATH, "eye_size")
mouth_info = load_expression_model(MOUTH_MODEL_PATH, "mouth_size")

cap = cv2.VideoCapture(str(VIDEO_PATH))
writer = None
rows = []
RUN_DIR = None
completed = False
window = "Expressions | Space pause | 1 left ROI | 2 right ROI | 3 mouth ROI | Q quit"

try:
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open: {VIDEO_PATH}")
    success, first = cap.read()
    if not success:
        raise RuntimeError("Cannot read the first frame.")
    height, width = first.shape[:2]
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    if not np.isfinite(fps) or fps <= 0:
        fps = 30.0
        print("Invalid video FPS. Using 30 FPS, so playback may be out of sync.")
    previous_gray, points, head_corners, regions = initialize_expression_tracking(first)
    reference = points.copy()
    RUN_DIR = PROJECT_DIR / "expression_runs" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    RUN_DIR.mkdir(parents=True, exist_ok=False)
    panel_width = 320
    canvas_height = max(height, 390)
    # Use even frame dimensions so encoders do not cut off the last column.
    out_width = (width + panel_width + 1) // 2 * 2
    out_height = (canvas_height + 1) // 2 * 2
    if SAVE_DEBUG_VIDEO:
        writer = cv2.VideoWriter(str(RUN_DIR / "debug.mp4"),
                                 cv2.VideoWriter_fourcc(*"mp4v"), fps,
                                 (out_width, out_height))
        if not writer.isOpened():
            raise RuntimeError("Cannot create the debug video.")
    cv2.namedWindow(window, cv2.WINDOW_NORMAL)
    frame = first
    frame_index = 0
    tracking_active = True
    revision = 0

    while True:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        transform = None
        aligned = None
        if frame_index == 0:
            transform = np.eye(3, dtype=np.float64)
        elif tracking_active:
            current, ref = track_points(previous_gray, gray, points, reference)
            if current is None or len(current) < MIN_POINTS:
                tracking_active = False
            else:
                affine, status = cv2.estimateAffinePartial2D(
                    ref.reshape(-1, 2), current.reshape(-1, 2),
                    method=cv2.RANSAC, ransacReprojThreshold=RANSAC_THRESHOLD,
                    maxIters=2000, confidence=0.99, refineIters=10)
                if affine is None or status is None or not np.isfinite(affine).all():
                    tracking_active = False
                else:
                    inliers = status.ravel().astype(bool)
                    if inliers.sum() < MIN_POINTS:
                        tracking_active = False
                    else:
                        transform = np.vstack([affine, [0., 0., 1.]])
                        points, reference = current[inliers].copy(), ref[inliers].copy()
        if tracking_active:
            scale = np.linalg.norm(transform[:2, 0])
            corners = cv2.perspectiveTransform(head_corners, transform)
            xy = corners.reshape(-1, 2)
            geometry_ok = (np.isfinite(xy).all() and 0.05 < scale < 20
                           and (np.abs(xy[:, 0]) < 3 * width).all()
                           and (np.abs(xy[:, 1]) < 3 * height).all())
            if geometry_ok:
                geometry_ok = 100 < abs(cv2.contourArea(xy.astype(np.float32))) < width * height
            tracking_active = bool(geometry_ok)
        if tracking_active:
            points, reference = replenish(gray, points, reference, transform, corners)
            inverse = np.linalg.inv(transform)
            aligned = cv2.warpAffine(gray, inverse[:2], (width, height))
            valid_mask = cv2.warpAffine(np.full(gray.shape, 255, np.uint8),
                                       inverse[:2], (width, height), flags=cv2.INTER_NEAREST)

        canvas = np.zeros((out_height, out_width, 3), np.uint8)
        canvas[:height, :width] = frame
        if tracking_active:
            cv2.polylines(canvas, [np.rint(corners).astype(np.int32)], True, (255, 255, 0), 2)
        row = {"frame_index": frame_index, "time_s": frame_index / fps,
               "head_tracking_ok": tracking_active, "manual_revision": revision}

        for index, region in enumerate(regions):
            name = region["name"]
            state, valid, score, method = "unknown", False, None, "LOST"
            rect = region["rect"]
            crop = None
            if tracking_active:
                if name == "mouth":
                    # Keep the mouth region fixed in the aligned head frame. Do not search for a mouth template.
                    method = "HEAD_ROI"
                else:
                    rect, score, matched = locate_eye(aligned, valid_mask, region)
                    method = "MATCH" if matched else "PREDICTED"
                x, y, w, h = rect
                if x >= 0 and y >= 0 and x+w <= width and y+h <= height:
                    crop = aligned[y:y+h, x:x+w]
                    mask = valid_mask[y:y+h, x:x+w]
                    valid = (crop.shape == (h, w) and mask.shape == (h, w)
                             and crop.size > 0 and bool(np.all(mask > 250)))
                if valid:
                    state = classify_region(crop, mouth_info if name == "mouth" else eye_info)
                color = ((0,255,0) if state == "open" else (0,0,255)
                         if state == "closed" else (0,165,255))
                polygon = cv2.perspectiveTransform(rect_points(rect), transform)
                cv2.polylines(canvas, [np.rint(polygon).astype(np.int32)], True, color, 2)
            row[name] = state
            row[name + "_crop_valid"] = valid
            row[name + "_localization"] = method
            row[name + "_ssd"] = float(score) if score is not None and np.isfinite(score) else None
            row[name + "_rect"] = list(rect)
            top = index * 130
            cv2.putText(canvas, f"{name}: {state}", (width+8,top+22),
                        cv2.FONT_HERSHEY_SIMPLEX,0.43,(255,255,255),1)
            cv2.putText(canvas, method, (width+8,top+43),
                        cv2.FONT_HERSHEY_SIMPLEX,0.43,(255,255,255),1)
            if valid:
                ratio = min(290/crop.shape[1], 75/crop.shape[0])
                preview = cv2.resize(crop, (max(1,int(crop.shape[1]*ratio)),
                                             max(1,int(crop.shape[0]*ratio))))
                ph,pw = preview.shape
                canvas[top+50:top+50+ph, width+8:width+8+pw] = cv2.cvtColor(preview,cv2.COLOR_GRAY2BGR)
        if not tracking_active:
            cv2.putText(canvas,"Head tracking lost: expressions unknown",(10,30),
                        cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,0,255),2)
        rows.append(row)
        if writer is not None:
            writer.write(canvas)
        cv2.imshow(window,canvas)
        key = cv2.waitKey(max(1,round(1000/fps))) & 255
        if key == ord(" "):
            while True:
                key = cv2.waitKey(30) & 255
                if key in (ord(" "),ord("1"),ord("2"),ord("3"),ord("q"),27):
                    break
                if cv2.getWindowProperty(window,cv2.WND_PROP_VISIBLE)<1:
                    key=ord("q");break
        if key in (ord("1"),ord("2"),ord("3")) and tracking_active:
            region = regions[key-ord("1")]
            rect = select_roi(cv2.cvtColor(aligned,cv2.COLOR_GRAY2BGR),
                              "Reselect region, then Enter")
            x,y,w,h = rect
            region.update(rect=rect,template=aligned[y:y+h,x:x+w].copy())
            revision += 1
            print(f"Frame {frame_index}: {region['name']} updated. The change starts on the next frame.")
        if key in (ord("q"),27) or cv2.getWindowProperty(window,cv2.WND_PROP_VISIBLE)<1:
            break
        previous_gray = gray.copy()
        success, frame = cap.read()
        if not success:
            completed = True
            break
        frame_index += 1
finally:
    cap.release()
    if writer is not None:
        writer.release()
    cv2.destroyAllWindows()
    # Keep processed frames if playback stops early. The player shows a neutral expression for unprocessed frames.
    if RUN_DIR is not None and rows:
        with open(RUN_DIR / "expressions.csv", "w", newline="",encoding="utf-8-sig") as file:
            csv_writer = csv.DictWriter(file,fieldnames=list(rows[0]))
            csv_writer.writeheader();csv_writer.writerows(rows)
        payload = {"schema_version":1,"source_video":VIDEO_PATH.name,
                   "fps":fps,"completed":completed,"rows":rows}
        with open(RUN_DIR / "expressions.json","w",encoding="utf-8") as file:
            json.dump(payload,file,ensure_ascii=False,allow_nan=False)
        with open(RUN_DIR / "settings.json","w",encoding="utf-8") as file:
            json.dump({"eye_model":str(EYE_MODEL_PATH),"mouth_model":str(MOUTH_MODEL_PATH),
                       "initialization":"manual head and regions",
                       "mouth_localization":"fixed head-aligned ROI",
                       "eye_localization":"initial template SSD with head fallback",
                       "fps":fps,"timestamp":"frame_index / fps; assumes constant frame rate",
                       "feature_params":FEATURE_PARAMS,"lk_params":LK_PARAMS,
                       "fb_threshold":FB_THRESHOLD,"ransac_threshold":RANSAC_THRESHOLD,
                       "search_margin":SEARCH_MARGIN,"match_threshold":MATCH_THRESHOLD},
                      file,ensure_ascii=False,indent=2)
        print("Saved to:",RUN_DIR)


ValueError: Unknown class mapping: {0: 'closed', 1: 'a', 2: 'i', 3: 'u', 4: 'e', 5: 'o'}

## 3D Playback
The next cell opens the player in your browser. Select the VRM file, the same source video and expressions.json.
For a front-facing video without mirroring, the left eye on screen maps to the avatar’s right eye. Turn on the eye swap option for a mirrored video.
Turn smoothing off first to see the raw predictions. Then turn it on to compare the results.

In [ ]:
# Start the server in the background so this cell can finish and later cells can run.
# On the page, select a local VRM file, the source video and the new expressions.json file.
from IPython.display import IFrame, display

PREVIEW_DIR = Path.cwd().resolve()
if not (PREVIEW_DIR / "video_driver.html").is_file():
    raise FileNotFoundError("Keep this notebook and video_driver.html in the same working folder.")
if "expression_preview_server" in globals():
    expression_preview_server.shutdown()
    expression_preview_server.server_close()
expression_preview_server = ThreadingHTTPServer(
    ("127.0.0.1", 0),
    partial(SimpleHTTPRequestHandler, directory=str(PREVIEW_DIR)))
expression_preview_thread = Thread(target=expression_preview_server.serve_forever,daemon=True)
expression_preview_thread.start()
expression_preview_url = f"http://127.0.0.1:{expression_preview_server.server_port}/video_driver.html"
print("Browser preview:",expression_preview_url)
webbrowser.open(expression_preview_url)
display(IFrame(expression_preview_url,width="100%",height=850))


In [ ]:
# Leave this False so Run All keeps the preview open. Set it to True to stop the server.
STOP_PREVIEW = False
if STOP_PREVIEW and "expression_preview_server" in globals():
    expression_preview_server.shutdown()
    expression_preview_server.server_close()
    del expression_preview_server
    print("Preview server stopped.")
